# filename · A — Gemma 4 12B QAT 품질 실험

**설치 → Google Drive 연결 → 데이터 읽기 → 설정 변경 → 실행·Drive 저장** 순서입니다.

평가 사진·원본 영상·정답은 팀이 준비합니다. 영상 정답은 영상 한 편당 하나입니다. 원본 영상을 올리면 FFmpeg로 대표 프레임을 추출합니다. 평가 사진과 정답은 합성하지 않습니다.
처음에는 `MOCK=True`로 파일 읽기와 리포트 생성을 확인하고, 실제 평가는 `False`로 바꾸세요.

선택 모델의 LM Studio 표시명은 `google/gemma-4-12b-qat`입니다. A는 Google의 **QAT 학습을 거친 BF16 체크포인트**를 사용합니다. B에서 실행할 Q4 GGUF와 정밀도가 다르므로 최종 품질은 B에서도 재확인하세요.
[모델과 파일 구분](https://github.com/yooongZa/filename-model-quality-bench/blob/main/docs/MODEL.md)

Based-On: none — 직접 대응하는 LMS 예제가 없는 project_extension입니다.

## 1. 설치

Colab에서 실제 추론할 때는 **런타임 → 런타임 유형 변경 → GPU(A100)** 를 선택하세요.
이 셀은 Colab에서 코드를 받고 필요한 패키지를 설치합니다. 모델 가중치는 실제 실행 단계에서 받습니다.
로컬에서 모의 실행할 때는 저장소 폴더에서 `pip install -r requirements.txt`를 먼저 실행하세요.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import shutil
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False

if IN_COLAB:
    # 원본 영상에서 프레임을 추출할 도구가 없을 때만 설치합니다.
    if not all(shutil.which(tool) for tool in ("ffmpeg", "ffprobe")):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
    PROJECT = Path("/content/filename-model-quality-bench")
    if not PROJECT.exists():
        subprocess.run([
            "git", "clone", "https://github.com/yooongZa/filename-model-quality-bench.git", str(PROJECT)
        ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements-colab.txt")], check=True)
else:
    # 로컬에서는 현재 폴더 또는 상위 폴더에서 저장소를 찾습니다.
    PROJECT = Path.cwd()
    if not (PROJECT / "a_quality.py").exists():
        PROJECT = PROJECT.parent

assert (PROJECT / "a_quality.py").exists(), "filename-model-quality-bench 폴더에서 실행해 주세요."
sys.path.insert(0, str(PROJECT))
print("코드 위치:", PROJECT)

## 2. Google Drive 연결과 데이터 준비

아래 폴더를 ZIP으로 묶어 Google Drive의 **내 드라이브 → filename → datasets → evaluation.zip**에 올리세요. 사진만 또는 원본 영상만 준비해도 됩니다. 프레임은 코드가 추출합니다.

```text
evaluation/
  images/
    city.jpg
    picnic.jpg
  videos/
    walking.mp4
  answers/
    city.json
    picnic.json
    walking.json
```

- `images/`: 사진 한 장이 평가 한 건입니다.
- `frames/영상이름/`는 이미 추출한 프레임을 직접 사용하고 싶을 때만 준비합니다.
- `videos/`: MP4·MOV·M4V·MKV·WEBM·AVI 원본을 넣으면 대표 프레임을 추출합니다. FFmpeg와 ffprobe가 필요합니다.
- 기본은 10초당 1장·최대 5장이며, 5장이면 10%·30%·50%·70%·90% 시점입니다.
- 추출 간격과 최대 장수는 바로 아래 업로드 셀에서 바꿉니다. 기존 JPG·PNG·WEBP·BMP도 사용할 수 있습니다.
- 영상 한 편의 정답은 `answers/영상ID.json` 한 개입니다. 예: `videos/walking.mp4` → `answers/walking.json`. 영상 전체의 태그·근거를 작성하며 프레임별 정답은 필요하지 않습니다. 정답 없이도 형식·속도 검사는 가능합니다.
- Colab에는 사진이 업로드되므로, 팀에서 Colab 사용이 가능한 자료를 선택하세요.

[데이터 형식과 정답 작성법](https://github.com/yooongZa/filename-model-quality-bench/blob/main/docs/DATASET.md)

### 아래 경로 설정만 바꾸면 됩니다

- `USE_DRIVE=True`: Drive의 ZIP을 읽고 결과를 Drive에 저장합니다. Colab 기본값입니다.
- `DRIVE_ROOT`: 데이터와 결과를 보관할 Drive 폴더입니다.
- `DRIVE_ZIP`: 직접 업로드한 평가 ZIP의 경로입니다.
- `USE_DRIVE=False`: 기존 PC ZIP 업로드 방식입니다. 결과는 Colab 임시 디스크에 저장됩니다.

Drive 연결 창에서 계정 접근을 허용하세요. 연결에 실패하면 셀이 중단됩니다.
원본 ZIP은 Drive에 그대로 두고, 압축 해제·이미지 읽기는 Colab 디스크에서 처리합니다.
[Drive 저장·재개 안내](https://github.com/yooongZa/filename-model-quality-bench/blob/main/docs/COLAB_DRIVE.md)

정답은 `keywords`, `reason`, `place_name` 3개 필드입니다. 허용 태그 개수에는 제한이 없고 모델 출력은 최대 5개입니다. 관찰 가능한 표현을 충분히 적되 100개를 채울 필요는 없습니다. [정답 작성 예시](https://github.com/yooongZa/filename-model-quality-bench/blob/main/docs/DATASET.md)


In [ ]:
# 기본값: Colab에서는 Google Drive, 로컬에서는 기존 ZIP 가져오기
USE_DRIVE = IN_COLAB
DRIVE_ROOT = Path("/content/drive/MyDrive/filename")
DRIVE_ZIP = DRIVE_ROOT / "datasets" / "evaluation.zip"

if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive 연결은 Colab에서 실행해 주세요.")
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive 연결을 확인해 주세요.")
    RESULTS_ROOT = DRIVE_ROOT / "runs"
else:
    RESULTS_ROOT = PROJECT / "runs"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("입력 ZIP:", DRIVE_ZIP if USE_DRIVE else "다음 셀에서 PC ZIP 선택")
print("결과 저장 폴더:", RESULTS_ROOT)

In [ ]:
import io
import uuid
from filename_bench.uploads import import_dataset_zip
from filename_bench.data import load_cases
from filename_bench.colab_storage import prepare_drive_dataset

# 원본 영상에서 추출할 후보 수를 정합니다. Python 영상 패키지는 필요하지 않습니다.
SECONDS_PER_FRAME = 10  # 몇 초당 한 장을 추출할지
MAX_EXTRACTED_FRAMES = 5  # 영상 한 개에서 뽑을 최대 장수
VIDEO_OPTIONS = {"seconds_per_frame": SECONDS_PER_FRAME, "max_frames": MAX_EXTRACTED_FRAMES}

# Drive 사용 시 아래 두 값은 비워 둡니다. USE_DRIVE=False일 때만 사용합니다.
LOCAL_ZIP = ""          # 로컬 실행 예: "/내폴더/evaluation.zip"
EXISTING_DATASET = ""   # 이미 가져온 데이터가 있으면 아래 출력된 데이터 경로를 입력합니다.

if USE_DRIVE:
    DATASET = prepare_drive_dataset(
        DRIVE_ZIP, PROJECT / "data" / "drive_cache", video_options=VIDEO_OPTIONS
    )
elif EXISTING_DATASET:
    DATASET = Path(EXISTING_DATASET).expanduser().resolve()
else:
    if LOCAL_ZIP:
        archive = Path(LOCAL_ZIP).expanduser()
    elif IN_COLAB:
        from google.colab import files
        uploaded = files.upload()  # 파일 선택 창에서 ZIP 한 개를 선택하세요.
        if len(uploaded) != 1:
            raise ValueError("평가 ZIP 한 개를 선택해 주세요.")
        archive = io.BytesIO(next(iter(uploaded.values())))
    else:
        raise ValueError("LOCAL_ZIP에 준비한 ZIP의 경로를 입력해 주세요.")

    # 다시 업로드해도 기존 데이터에 덮어쓰지 않도록 새 폴더를 만듭니다.
    destination = PROJECT / "data" / "uploads" / uuid.uuid4().hex[:12]
    DATASET = import_dataset_zip(archive, destination, video_options=VIDEO_OPTIONS)

cases, _ = load_cases(DATASET, video_options=VIDEO_OPTIONS)
print("데이터 경로:", DATASET)
for case in cases[:20]:
    print(case["id"], "|", case["media_type"], "| 이미지", len(case["paths"]), "장")
print("전체 평가 단위:", len(cases), "건")

## 3. 바꿔 볼 설정

데이터 경로를 정한 뒤에는 **이 셀에서 실험 조건을 수정**합니다. 한 번에 한 가지 조건을 바꾸세요.
`SUITE`는 실험 묶음입니다. `smoke`로 확인한 뒤 `resolution`, `vision`, `output`, `frames`, `selection` 중 하나를 실행합니다.

`frames` 실험의 기본 비교값은 1·3·6·12장, `selection`은 후보 중 6장을 선택합니다. 보유한 프레임 수에 맞춰 비교값을 바꾸거나 충분한 후보를 올리세요. 부족한 프레임은 `unsupported`로 기록합니다.

`RESUME_RUN_NAME`을 비워 두면 새 실험을 만듭니다. 이어서 실행할 때는 Drive의 `runs/` 아래에 표시된 기존 폴더 이름을 넣고 **이전과 같은 조건**을 사용하세요. 서로 다른 노트북에서 같은 실험 폴더를 동시에 사용하지 마세요.


In [ ]:
from datetime import datetime
from filename_bench.common import load_config
from filename_bench.runner import run

MOCK = True                 # True: GPU 없이 흐름 확인 / False: 실제 모델 평가
SUITE = "smoke"             # smoke, baseline, resolution, vision, output, frames, selection
CASE_IDS = None             # None: 업로드 전체 / 일부만 평가하려면 ["city", "cycling"]
SHORT_EDGE_PX = 1080         # 입력 이미지의 짧은 변. 원본보다 확대하지 않습니다.
VISION_TOKEN_BUDGET = 1120  # 이미지 한 장의 시각 토큰 상한: 70, 140, 280, 560, 1120
VIDEO_FRAMES = "all"        # 추출 결과 모두 사용. 후보 중 일부만 넣으려면 1, 3 등으로 수정
MAX_NEW_TOKENS = 256        # 답변에 허용할 최대 토큰 수. JSON이 잘리면 늘리세요.
CONTEXT_LENGTH = 32768      # 입력과 출력 토큰을 합친 상한
MAX_NEW_REQUESTS = None     # None: 모두 / 2: 이번 실행은 최대 두 요청
RESUME_RUN_NAME = ""        # 재개 예: "20261015_210000_123456" / 새 실험은 빈 문자열
if RESUME_RUN_NAME and (RESUME_RUN_NAME in {".", ".."} or any(c in RESUME_RUN_NAME for c in "/\\:")):
    raise ValueError("RESUME_RUN_NAME에는 runs 아래 폴더 이름만 입력하세요.")
RUN_NAME = RESUME_RUN_NAME or datetime.now().strftime("%Y%m%d_%H%M%S_%f")
if RESUME_RUN_NAME and not (RESULTS_ROOT / RUN_NAME).is_dir():
    raise FileNotFoundError(f"이어서 실행할 결과 폴더가 없습니다: {RESULTS_ROOT / RUN_NAME}")

config = load_config(PROJECT / "configs/a_quality.json")
config["video_extraction"].update(VIDEO_OPTIONS)
config.update(dataset_root=str(DATASET), run_dir=str(RESULTS_ROOT / RUN_NAME), case_ids=CASE_IDS)
config["baseline"].update(
    short_edge_px=SHORT_EDGE_PX, vision_token_budget=VISION_TOKEN_BUDGET,
    frames=VIDEO_FRAMES, max_new_tokens=MAX_NEW_TOKENS, context_per_request=CONTEXT_LENGTH,
)

# 자동 비교할 후보 목록입니다. 해당 SUITE에서 이 값 하나만 바꿔 반복합니다.
config["sweeps"]["short_edge_px"] = [1080, 720, 480, 360, 240]
config["sweeps"]["vision_token_budget"] = [1120, 560, 280, 140, 70]
config["sweeps"]["max_new_tokens"] = [512, 256, 128]
config["sweeps"]["frames"] = [1, 3, 6, 12]  # 세 프레임만 있으면 [1, 3]으로 수정

print("모델:", config["backend"]["model_label"])
print("실제 로드할 체크포인트:", config["backend"]["model_id"])
print("추론 정밀도: BF16 / QAT 학습 체크포인트")
print("결과 위치:", config["run_dir"])

### 실행 전 확인

모델을 로드하지 않고 입력 개수와 요청 수를 확인합니다. 사진만 올렸다면 영상용 조건은 건너뜁니다.
원본 영상의 추출과 캐시 확인은 모델 추론 전에 실행됩니다. 정답이 없으면 허용 태그 일치율은 비워 둡니다. 정답 유무와 관계없이 최종 품질은 `review.csv`에서 사람이 평가합니다.

In [ ]:
plan = run(config, "a", suite=SUITE, mock=MOCK, dry_run=True)

## 4. 실행

`MOCK=False`이면 모델을 다운로드하고 실제 추론합니다. Hugging Face 모델 페이지에서 약관 동의가 필요할 수 있습니다. 접근 오류가 나면 계정 권한을 확인하고 Colab 보안 비밀에 `HF_TOKEN`을 등록하세요. 토큰을 코드에 직접 적지 마세요.

이 실행 셀을 다시 실행하면 같은 설정에서 미완료 요청만 이어갑니다. `RESUME_RUN_NAME`이 비어 있는 상태에서 설정 셀을 다시 실행하면 새 실험이 됩니다.

Drive 모드에서는 요청 한 건이 끝날 때마다 원문·측정 기록을 Drive에 씁니다. HTML·CSV는 실행 완료 또는 처리 가능한 중단 시 갱신합니다. 갑자기 런타임이 종료되면 마지막 기록과 Drive 동기화가 끝나지 않을 수 있습니다.
저장 오류는 화면에 표시되며, 조용히 임시 디스크 저장으로 전환하지 않습니다. 재개는 동일한 데이터·설정·코드·모델일 때 가능합니다.


In [ ]:
result = run(
    config, "a", suite=SUITE, mock=MOCK,
    resume=True, limit=MAX_NEW_REQUESTS,
)
print(result)
if result["stopped_on_error"]:
    print("실행 오류로 중단했습니다. 결과 폴더의 events.jsonl과 report.html을 확인하세요.")

## 5. 결과 확인

- `report.html`: 조건별 성공률·시간·응답 원문
- `summary.csv`: 조건별 요약 / `requests.jsonl`: 요청별 원문
- `review.csv`: 태그 정확성·파일명 활용도·근거 없는 추정 등을 사람이 평가
- `config.json`, `backend_*.json`: 설정·모델·실행 환경 기록

모의 실행 화면에는 **모델 성능 결과 아님**이 표시됩니다. 실제 응답의 JSON 성공률과 내용 정확도를 따로 보세요.

Drive 모드의 기본 결과 위치: `내 드라이브/filename/runs/<실험이름>/<SUITE>/`. 모의 실행 폴더에는 `_mock`이 붙습니다. 원본 데이터와 추출 프레임은 결과 폴더에 포함하지 않습니다.

**허용 태그 일치율은 참고값**입니다. 계산은 목록에 포함된 출력 태그 수 / 출력 태그 수이며, 허용 목록 100개 중 출력 5개가 모두 포함되면 100%입니다. `metrics.csv`와 HTML에 일치·미등록 태그가, `summary.csv`에 조건별 평균과 계산 건수가 저장됩니다. 최종 품질은 사람이 `review.csv`의 정확성·파일명 활용도(각 0~2점)를 평가합니다. 재현율은 계산하지 않습니다.


In [ ]:
from IPython.display import HTML, display

run_directory = Path(result["run_dir"])
report_path = run_directory / "report.html"
print("리포트:", report_path)
display(HTML(report_path.read_text(encoding="utf-8")))

### 결과 ZIP 저장과 선택 다운로드

결과 파일은 이미 위 결과 폴더에 저장되어 있습니다. 아래 셀은 공유용 ZIP을 같은 위치에 추가합니다.
Drive 모드에서는 ZIP도 Drive에 남습니다. PC에도 받으려면 `DOWNLOAD_TO_PC=True`로 바꾸세요.
원본 사진·영상·추출 프레임은 포함하지 않습니다. 실행 설정에는 데이터 경로가 기록되므로 외부 공유 전 확인하세요.

In [ ]:
import shutil

DOWNLOAD_TO_PC = False  # Drive 보관만 하면 False, PC에도 받으려면 True
result_zip = shutil.make_archive(str(run_directory) + "_report", "zip", run_directory)
print("결과 ZIP 저장 완료:", result_zip)
if IN_COLAB and DOWNLOAD_TO_PC:
    from google.colab import files
    files.download(result_zip)